# Entregável 1 — Relatório de Aquisição dos Biossinais

**Disciplina:** Aquisição de Biossinais  
**Equipe:** José Lessa & Matheus Rocha  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL (PhysioNet)  

---

## Objetivo

Este notebook documenta a etapa de **Aquisição dos Biossinais** do pipeline da disciplina, 
descrevendo o dataset PTB-XL, seu hardware de origem, protocolo experimental, taxas de 
amostragem e realizando a visualização dos sinais brutos de ECG com identificação de 
possíveis problemas (saturação, drift, ruído de linha).

## 1. Configuração e Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import wfdb
from pathlib import Path

# Configurações de plot
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Diretório de dados
DATA_DIR = Path('../../data/ptb-xl')
FIG_DIR = Path('../figuras')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Configuração concluída.')

## 2. Download do Dataset PTB-XL

O dataset PTB-XL está disponível no PhysioNet: https://physionet.org/content/ptb-xl/1.0.3/

**Atenção:** O download completo é ~2.4 GB. A célula abaixo faz o download automaticamente 
usando a biblioteca `wfdb`. Se você já baixou o dataset, pule esta célula.

In [ ]:
# Descomente a linha abaixo para baixar o dataset completo (~2.4 GB)
# AVISO: Pode demorar bastante dependendo da conexão.

if not DATA_DIR.exists():
    print('Baixando PTB-XL do PhysioNet...')
    print('Isso pode levar vários minutos (~2.4 GB).')
    wfdb.dl_database('ptb-xl/1.0.3', dl_dir=str(DATA_DIR))
    print('Download concluído!')
else:
    print(f'Dataset já encontrado em: {DATA_DIR.resolve()}')

## 3. Descrição do Hardware e Protocolo de Aquisição

### 3.1 Tipo de Biossinal

**Eletrocardiograma (ECG)** de **12 derivações** clínicas padrão:
- 6 derivações dos membros: I, II, III, aVR, aVL, aVF
- 6 derivações precordiais: V1, V2, V3, V4, V5, V6

### 3.2 Hardware/Sensor

Os registros foram coletados com equipamentos clínicos da **Schiller AG**, modelo de 
eletrocardiógrafo hospitalar utilizado no **Helios Klinikum de Berlim**, em parceria com 
o **Physikalisch-Technische Bundesanstalt (PTB)**.

Características do equipamento:
- Resolução de digitalização: 16 bits
- Faixa dinâmica adequada para sinais cardíacos (±5 mV tipicamente)
- Eletrodos de superfície padrão (Ag/AgCl)

### 3.3 Taxa de Amostragem e Justificativa (Nyquist)

O dataset disponibiliza dois formatos:
- **500 Hz** — Alta resolução (original)
- **100 Hz** — Versão reamostrada

**Justificativa via Teorema de Nyquist:**

O sinal de ECG contém informação relevante na faixa de **0.05 Hz a 150 Hz** (recomendação 
da AHA/ACC para ECG diagnóstico). Pelo teorema de Nyquist, a taxa de amostragem mínima é:

$$ f_s \geq 2 \times f_{max} = 2 \times 150 = 300 \text{ Hz} $$

A taxa de **500 Hz** atende com folga o critério de Nyquist, permitindo reconstrução 
fidedigna do sinal sem aliasing. A versão de 100 Hz preserva a morfologia principal 
(ondas P, QRS, T) mas perde detalhes de alta frequência.

### 3.4 Protocolo Experimental

- **Duração por registro:** 10 segundos
- **Posição do paciente:** Decúbito dorsal (supina), protocolo hospitalar padrão
- **Condições:** Ambiente clínico controlado (hospital)
- **Período de coleta:** 1989 a 1996
- **Local:** Helios Klinikum, Berlim, Alemanha

### 3.5 Ambiente de Aquisição

Aquisição realizada em ambiente hospitalar clínico, com equipamentos calibrados 
e protocolos padronizados de ECG de repouso de 12 derivações.

## 4. Carregamento e Exploração dos Metadados

In [ ]:
# Carregar metadados do dataset
metadata_path = DATA_DIR / 'ptbxl_database.csv'
df = pd.read_csv(metadata_path, index_col='ecg_id')

print(f'Total de registros: {len(df)}')
print(f'Colunas disponíveis: {list(df.columns)}')
print(f'\nPrimeiras linhas:')
df.head()

In [ ]:
# Estatísticas descritivas dos metadados demográficos
print('=== Distribuição por Sexo ===')
print(df['sex'].value_counts())

print('\n=== Estatísticas de Idade ===')
print(df['age'].describe())

print(f'\n=== Dispositivos de Aquisição ===')
print(df['device'].value_counts())

print(f'\n=== Período de Aquisição ===')
print(f"Data mais antiga: {df['recording_date'].min()}")
print(f"Data mais recente: {df['recording_date'].max()}")

In [ ]:
# Visualização da distribuição de idades
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de idades
axes[0].hist(df['age'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Idade (anos)')
axes[0].set_ylabel('Número de registros')
axes[0].set_title('Distribuição de Idades dos Pacientes')
axes[0].axvline(df['age'].mean(), color='red', linestyle='--', label=f"Média: {df['age'].mean():.1f}")
axes[0].legend()

# Distribuição por sexo
sex_counts = df['sex'].value_counts()
sex_labels = ['Masculino' if s == 0 else 'Feminino' for s in sex_counts.index]
colors = ['#3498db', '#e74c3c']
axes[1].bar(sex_labels, sex_counts.values, color=colors, edgecolor='white', alpha=0.8)
axes[1].set_ylabel('Número de registros')
axes[1].set_title('Distribuição por Sexo')
for i, v in enumerate(sex_counts.values):
    axes[1].text(i, v + 100, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'distribuicao_demografica.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em:', FIG_DIR / 'distribuicao_demografica.png')

## 5. Visualização dos Sinais Brutos de ECG

Nesta seção, carregamos e visualizamos registros de ECG bruto das 12 derivações.

In [ ]:
def carregar_ecg(ecg_id, df, data_dir, sampling_rate=500):
    """
    Carrega um registro de ECG do PTB-XL.
    
    Parâmetros:
        ecg_id: ID do registro (inteiro)
        df: DataFrame com metadados
        data_dir: diretório raiz do dataset
        sampling_rate: 100 ou 500 Hz
    
    Retorna:
        record: objeto wfdb.Record com o sinal
    """
    if sampling_rate == 500:
        filename_col = 'filename_hr'
    else:
        filename_col = 'filename_lr'
    
    filename = df.loc[ecg_id, filename_col]
    record = wfdb.rdrecord(str(data_dir / filename))
    return record


def plotar_ecg_12_derivacoes(record, titulo='', salvar_como=None):
    """
    Plota as 12 derivações de um ECG em grade 6x2.
    
    Parâmetros:
        record: objeto wfdb.Record
        titulo: título do gráfico
        salvar_como: caminho para salvar a figura (opcional)
    """
    signal = record.p_signal
    fs = record.fs
    channels = record.sig_name
    t = np.arange(signal.shape[0]) / fs
    
    fig, axes = plt.subplots(6, 2, figsize=(18, 16), sharex=True)
    fig.suptitle(titulo, fontsize=16, fontweight='bold', y=1.01)
    
    for i, ax in enumerate(axes.flat):
        if i < len(channels):
            ax.plot(t, signal[:, i], color='#2c3e50', linewidth=0.7)
            ax.set_ylabel(f'{channels[i]}\n(mV)', fontsize=10)
            ax.set_title(f'Derivação {channels[i]}', fontsize=11, fontweight='bold', loc='left')
            
            # Fundo estilo papel milimetrado
            ax.set_facecolor('#fefefe')
            ax.grid(True, which='major', alpha=0.4, color='#e74c3c', linewidth=0.5)
            ax.grid(True, which='minor', alpha=0.15, color='#e74c3c', linewidth=0.3)
            ax.minorticks_on()
        else:
            ax.set_visible(False)
    
    axes[-1, 0].set_xlabel('Tempo (s)', fontsize=12)
    axes[-1, 1].set_xlabel('Tempo (s)', fontsize=12)
    
    plt.tight_layout()
    
    if salvar_como:
        plt.savefig(salvar_como, dpi=150, bbox_inches='tight')
        print(f'Figura salva em: {salvar_como}')
    
    plt.show()


print('Funções definidas.')

In [ ]:
# Carregar e plotar um registro NORMAL (NORM)
# Selecionar um registro com diagnóstico normal
ecg_id_normal = df.index[0]  # Primeiro registro como exemplo
record_normal = carregar_ecg(ecg_id_normal, df, DATA_DIR, sampling_rate=500)

print(f'Registro ECG ID: {ecg_id_normal}')
print(f'Frequência de amostragem: {record_normal.fs} Hz')
print(f'Duração: {record_normal.sig_len / record_normal.fs:.1f} s')
print(f'Número de canais: {record_normal.n_sig}')
print(f'Nome dos canais: {record_normal.sig_name}')
print(f'Unidades: {record_normal.units}')
print(f'Número de amostras: {record_normal.sig_len}')

plotar_ecg_12_derivacoes(
    record_normal,
    titulo=f'ECG Bruto — Registro #{ecg_id_normal} (500 Hz, 12 derivações)',
    salvar_como=FIG_DIR / 'ecg_bruto_exemplo_1.png'
)

In [ ]:
# Plotar mais alguns registros para diversidade de exemplos
# Selecionar registros em posições variadas do dataset
sample_ids = [df.index[100], df.index[500], df.index[1000]]

for ecg_id in sample_ids:
    record = carregar_ecg(ecg_id, df, DATA_DIR, sampling_rate=500)
    row = df.loc[ecg_id]
    
    info = f'Idade: {row["age"]:.0f}' if pd.notna(row.get('age')) else ''
    
    plotar_ecg_12_derivacoes(
        record,
        titulo=f'ECG Bruto — Registro #{ecg_id} ({info})',
        salvar_como=FIG_DIR / f'ecg_bruto_registro_{ecg_id}.png'
    )

## 6. Identificação Visual de Possíveis Problemas

Nesta seção, investigamos possíveis problemas nos sinais brutos:
- **Saturação:** amplitude que atinge o limite do conversor A/D
- **Drift de baseline:** deslocamento lento da linha de base
- **Ruído de linha (50/60 Hz):** interferência da rede elétrica

In [ ]:
def analisar_qualidade_sinal(record, ecg_id):
    """
    Analisa indicadores básicos de qualidade de um registro de ECG.
    
    Verifica:
    - Saturação (valores no limite)
    - Drift de baseline (tendência linear)
    - Amplitude do sinal
    - Estatísticas básicas
    """
    signal = record.p_signal
    fs = record.fs
    channels = record.sig_name
    
    print(f'\n{"="*60}')
    print(f'ANÁLISE DE QUALIDADE — Registro #{ecg_id}')
    print(f'{"="*60}')
    
    for i, ch in enumerate(channels):
        sig = signal[:, i]
        
        # Estatísticas básicas
        amplitude_pp = np.max(sig) - np.min(sig)  # pico-a-pico
        media = np.mean(sig)
        std = np.std(sig)
        
        # Verificar saturação (valores muito próximos do máximo/mínimo por >10 amostras consecutivas)
        val_max = np.max(sig)
        val_min = np.min(sig)
        threshold = 0.001  # 1 uV de tolerância
        saturacao_alta = np.sum(np.abs(sig - val_max) < threshold)
        saturacao_baixa = np.sum(np.abs(sig - val_min) < threshold)
        tem_saturacao = (saturacao_alta > 10) or (saturacao_baixa > 10)
        
        # Verificar drift de baseline (diferença entre média da primeira e última metade)
        half = len(sig) // 2
        media_primeira = np.mean(sig[:half])
        media_segunda = np.mean(sig[half:])
        drift = abs(media_segunda - media_primeira)
        tem_drift = drift > 0.1  # >100 uV de drift
        
        # Verificar possível flat-line
        tem_flatline = std < 0.01
        
        status = ''
        if tem_saturacao:
            status += ' ⚠️ SATURAÇÃO'
        if tem_drift:
            status += ' ⚠️ DRIFT'
        if tem_flatline:
            status += ' ⚠️ FLAT-LINE'
        if not status:
            status = ' ✅ OK'
        
        print(f'{ch:>5s} | Amp: {amplitude_pp:6.3f} mV | '
              f'Média: {media:+7.4f} mV | '
              f'Std: {std:6.4f} mV | '
              f'Drift: {drift:6.4f} mV |{status}')


# Analisar os registros já carregados
record_1 = carregar_ecg(df.index[0], df, DATA_DIR, sampling_rate=500)
analisar_qualidade_sinal(record_1, df.index[0])

In [ ]:
# Analisar uma amostra maior para estatísticas gerais de qualidade
n_amostra = 200  # Analisar 200 registros aleatórios
np.random.seed(42)
ids_amostra = np.random.choice(df.index, size=n_amostra, replace=False)

resultados = []

for ecg_id in ids_amostra:
    try:
        record = carregar_ecg(ecg_id, df, DATA_DIR, sampling_rate=500)
        signal = record.p_signal
        
        for i, ch in enumerate(record.sig_name):
            sig = signal[:, i]
            half = len(sig) // 2
            
            resultados.append({
                'ecg_id': ecg_id,
                'canal': ch,
                'amplitude_pp': np.max(sig) - np.min(sig),
                'media': np.mean(sig),
                'std': np.std(sig),
                'drift': abs(np.mean(sig[half:]) - np.mean(sig[:half])),
                'max': np.max(sig),
                'min': np.min(sig)
            })
    except Exception as e:
        print(f'Erro no registro {ecg_id}: {e}')

df_qualidade = pd.DataFrame(resultados)
print(f'Analisados {n_amostra} registros ({len(df_qualidade)} canais no total).')
print('\n=== Estatísticas de Qualidade (todos os canais) ===')
print(df_qualidade[['amplitude_pp', 'std', 'drift']].describe())

In [ ]:
# Visualização da análise de qualidade
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribuição de amplitude pico-a-pico
axes[0].hist(df_qualidade['amplitude_pp'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Amplitude pico-a-pico (mV)')
axes[0].set_ylabel('Contagem')
axes[0].set_title('Distribuição de Amplitude (pico-a-pico)')
axes[0].axvline(df_qualidade['amplitude_pp'].median(), color='red', linestyle='--', 
                label=f"Mediana: {df_qualidade['amplitude_pp'].median():.2f} mV")
axes[0].legend()

# Distribuição de drift
axes[1].hist(df_qualidade['drift'], bins=50, color='#e67e22', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Drift de Baseline (mV)')
axes[1].set_ylabel('Contagem')
axes[1].set_title('Distribuição de Drift de Baseline')
axes[1].axvline(0.1, color='red', linestyle='--', label='Limiar (0.1 mV)')
axes[1].legend()

# Distribuição do desvio padrão
axes[2].hist(df_qualidade['std'], bins=50, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Desvio Padrão (mV)')
axes[2].set_ylabel('Contagem')
axes[2].set_title('Distribuição do Desvio Padrão')
axes[2].axvline(0.01, color='red', linestyle='--', label='Limiar flat-line (0.01 mV)')
axes[2].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'analise_qualidade_sinal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

In [ ]:
# Análise espectral para verificar ruído de 50/60 Hz
def plotar_espectro(record, ecg_id, derivacao_idx=1, salvar_como=None):
    """
    Plota o espectro de potência de uma derivação para identificar ruído de linha.
    """
    signal = record.p_signal[:, derivacao_idx]
    fs = record.fs
    ch_name = record.sig_name[derivacao_idx]
    
    # FFT
    n = len(signal)
    freqs = np.fft.rfftfreq(n, d=1/fs)
    fft_vals = np.abs(np.fft.rfft(signal)) ** 2
    fft_db = 10 * np.log10(fft_vals + 1e-12)  # em dB
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    
    # Sinal no tempo
    t = np.arange(n) / fs
    axes[0].plot(t, signal, color='#2c3e50', linewidth=0.7)
    axes[0].set_xlabel('Tempo (s)')
    axes[0].set_ylabel('Amplitude (mV)')
    axes[0].set_title(f'Sinal no Tempo — Derivação {ch_name} (Registro #{ecg_id})', fontweight='bold')
    
    # Espectro
    axes[1].plot(freqs, fft_db, color='#e74c3c', linewidth=0.7)
    axes[1].set_xlabel('Frequência (Hz)')
    axes[1].set_ylabel('Potência (dB)')
    axes[1].set_title(f'Espectro de Potência — Derivação {ch_name}', fontweight='bold')
    axes[1].set_xlim([0, 250])
    
    # Marcar frequência de rede (50 Hz na Europa)
    axes[1].axvline(50, color='orange', linestyle='--', alpha=0.8, label='50 Hz (rede elétrica)')
    axes[1].axvline(100, color='orange', linestyle=':', alpha=0.5, label='100 Hz (2º harmônico)')
    axes[1].axvline(150, color='orange', linestyle=':', alpha=0.5, label='150 Hz (3º harmônico)')
    axes[1].legend()
    
    plt.tight_layout()
    
    if salvar_como:
        plt.savefig(salvar_como, dpi=150, bbox_inches='tight')
        print(f'Figura salva em: {salvar_como}')
    
    plt.show()


# Análise espectral do primeiro registro
record_espectro = carregar_ecg(df.index[0], df, DATA_DIR, sampling_rate=500)
plotar_espectro(record_espectro, df.index[0], derivacao_idx=1,
               salvar_como=FIG_DIR / 'espectro_ecg_ruido_linha.png')

## 7. Comparação 500 Hz vs 100 Hz

In [ ]:
# Comparar a mesma derivação em 500 Hz e 100 Hz
ecg_id_comp = df.index[0]
record_500 = carregar_ecg(ecg_id_comp, df, DATA_DIR, sampling_rate=500)
record_100 = carregar_ecg(ecg_id_comp, df, DATA_DIR, sampling_rate=100)

derivacao = 1  # Derivação II (comum para análise)
ch_name = record_500.sig_name[derivacao]

t_500 = np.arange(record_500.sig_len) / record_500.fs
t_100 = np.arange(record_100.sig_len) / record_100.fs

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(t_500, record_500.p_signal[:, derivacao], color='#2c3e50', linewidth=0.7)
axes[0].set_ylabel('Amplitude (mV)')
axes[0].set_title(f'ECG Derivação {ch_name} — 500 Hz ({record_500.sig_len} amostras)', fontweight='bold')

axes[1].plot(t_100, record_100.p_signal[:, derivacao], color='#e74c3c', linewidth=0.7)
axes[1].set_ylabel('Amplitude (mV)')
axes[1].set_xlabel('Tempo (s)')
axes[1].set_title(f'ECG Derivação {ch_name} — 100 Hz ({record_100.sig_len} amostras)', fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'comparacao_500hz_100hz.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 8. Resumo e Conclusões

### Síntese do Entregável 1

| Item | Descrição |
|------|-----------|
| **Biossinal** | ECG de 12 derivações |
| **Hardware** | Schiller AG (equipamento hospitalar clínico) |
| **Amostragem** | 500 Hz (alta resolução), 100 Hz (reduzida) |
| **Nyquist** | 500 Hz > 2×150 Hz = 300 Hz ✅ |
| **Protocolo** | Decúbito dorsal, 10s por registro, ambiente hospitalar |
| **Local/Período** | Helios Klinikum, Berlim, 1989–1996 |
| **Volume** | 21.799 registros, 18.869 pacientes |
| **Formato** | WFDB (.hea + .dat) |

### Problemas Identificados

Os resultados da análise de qualidade das amostras avaliadas serão discutidos com base 
nos gráficos e métricas geradas acima. Os principais pontos a observar são:

1. **Saturação** — Verificar se existem canais com amplitude constante no limite
2. **Drift de baseline** — Presente em graus variáveis, esperado em ECG de repouso
3. **Ruído de 50 Hz** — Verificar presença de pico na análise espectral (rede elétrica europeia)
4. **Flat-line** — Canais com variância próxima de zero podem indicar desconexão

Esses achados serão tratados nos entregáveis subsequentes (SQI, Limpeza, Filtragem).